# ORI-C Notebook 01 — Synthetic Demo: Pre-threshold vs Cumulative Regime

This notebook demonstrates the two canonical ORI-C regimes on **synthetic data**:
- **Regime A (pre-threshold):** C(t) ≈ 0, no symbolic accumulation
- **Regime B (cumulative):** symbolic injection → threshold crossing → C(t) > 0 stable

All parameters are **fixed ex ante** per `PREREG_TEMPLATE.md`.

---
**Setup:** run from the repository root with the `cumulative_symbolic` conda environment active.
```bash
conda activate cumulative_symbolic
jupyter lab examples/notebook_01_synthetic_demo.ipynb
```

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../04_Code')

import numpy as np
import matplotlib.pyplot as plt

from pipeline.ori_c_pipeline import ORICConfig, run_oric

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

## 1. Configure and run the two reference regimes

We use `ORICConfig` (a **frozen dataclass** — all parameters fixed at construction, no mutation).
- `none` intervention → pre-threshold regime  
- `symbolic_injection` → cumulative regime

In [ ]:
SEED = 42
N_STEPS = 250

# Regime A — no symbolic intervention (pre-threshold)
cfg_A = ORICConfig(seed=SEED, n_steps=N_STEPS, intervention='none')
df_A  = run_oric(cfg_A)

# Regime B — symbolic injection at t0 = 30% of series
cfg_B = ORICConfig(seed=SEED, n_steps=N_STEPS, intervention='symbolic_injection')
df_B  = run_oric(cfg_B)

print(f'Regime A shape: {df_A.shape}')
print(f'Regime B shape: {df_B.shape}')
print(f'Columns: {list(df_A.columns)}')

## 2. Inspect the data

Each row is one time step. Key columns:

| Column | Description |
|--------|-------------|
| `t` | Time step |
| `O`, `R`, `I` | Internal state proxies |
| `Cap` | Capacity = O·R·I |
| `Sigma` / `demande_env` | Mismatch Σ(t) / demand |
| `S` | Symbolic stock |
| `C` | Order variable |
| `delta_C` | ΔC(t) — threshold detection variable |

In [ ]:
df_A[['t', 'O', 'R', 'I', 'Cap', 'S', 'C', 'delta_C']].head(8)

## 3. Detect the threshold

The detection criterion (pre-registered defaults):
$$\Delta C(t) > \mu_{\Delta C} + k \cdot \sigma_{\Delta C} \quad \text{for } m \text{ consecutive steps}$$
with $k = 2.5$, $m = 3$, baseline estimated on first `baseline_n = 30` points.

In [ ]:

K = 2.5
M = 3
BASELINE_N = 30

def detect_threshold(df, k=K, m=M, baseline_n=BASELINE_N):
    """Returns (threshold_value, first_index) or (None, None)."""
    dc = df['delta_C'].to_numpy(dtype=float)
    baseline = dc[:baseline_n]
    mu, sigma = np.nanmean(baseline), np.nanstd(baseline)
    thr = mu + k * sigma
    count = 0
    for i, v in enumerate(dc):
        if np.isfinite(v) and v > thr:
            count += 1
            if count >= m:
                return thr, i - m + 1
        else:
            count = 0
    return thr, None

thr_A, idx_A = detect_threshold(df_A)
thr_B, idx_B = detect_threshold(df_B)

print(f'Regime A — threshold level: {thr_A:.4f}, crossing index: {idx_A}')
print(f'Regime B — threshold level: {thr_B:.4f}, crossing index: {idx_B}')

## 4. Visualise C(t) and ΔC(t) for both regimes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
fig.suptitle('ORI-C: Pre-threshold (A) vs Cumulative (B) regimes', fontweight='bold', y=1.01)

colors = {'A': '#2563eb', 'B': '#7c3aed'}

for col_i, (label, df, thr, idx) in enumerate([
    ('A — Pre-threshold (no intervention)', df_A, thr_A, idx_A),
    ('B — Cumulative (symbolic injection)',  df_B, thr_B, idx_B),
]):
    c = colors['A'] if col_i == 0 else colors['B']

    # C(t)
    ax = axes[0, col_i]
    ax.plot(df['t'], df['C'], color=c, lw=1.5, label='C(t)')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    if idx is not None:
        ax.axvline(df['t'].iloc[idx], color='#dc2626', lw=1.5, ls=':', label=f'threshold at t={df["t"].iloc[idx]:.0f}')
    ax.set_title(f'C(t)  ·  {label}', fontsize=9)
    ax.set_ylabel('C(t)')
    ax.legend(fontsize=8)

    # ΔC(t)
    ax2 = axes[1, col_i]
    ax2.plot(df['t'], df['delta_C'], color=c, lw=1.2, alpha=0.8, label='ΔC(t)')
    ax2.axhline(thr, color='#dc2626', lw=1.5, ls='--', label=f'threshold = {thr:.3f}')
    ax2.set_title(f'ΔC(t)  ·  {label}', fontsize=9)
    ax2.set_ylabel('ΔC(t)')
    ax2.set_xlabel('t')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f'\nRegime A max_C = {df_A["C"].max():.4f}  |  Regime B max_C = {df_B["C"].max():.4f}')

## 5. ORI Core: O, R, I and Cap(t)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, (label, df) in zip(axes, [('A', df_A), ('B', df_B)]):
    ax.plot(df['t'], df['O'],   label='O(t)', alpha=0.8)
    ax.plot(df['t'], df['R'],   label='R(t)', alpha=0.8)
    ax.plot(df['t'], df['I'],   label='I(t)', alpha=0.8)
    ax.plot(df['t'], df['Cap'], label='Cap=O·R·I', lw=2, color='black')
    ax.set_title(f'ORI Core — Regime {label}', fontsize=10)
    ax.set_xlabel('t')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Verdict

The phase transition (Regime B) is unambiguously detected by the pre-registered criterion.
Regime A confirms the baseline: without symbolic injection, no threshold crossing occurs.
This is the expected **pre-threshold** behaviour — not a failure.

In [ ]:
for label, idx in [('A (pre-threshold)', idx_A), ('B (cumulative)', idx_B)]:
    if idx is None:
        verdict = 'INDETERMINATE — no threshold detected (expected for pre-threshold)'
    else:
        verdict = f'ACCEPT — threshold crossed at step {idx}'
    print(f'Regime {label}: {verdict}')

---
**Next:** see `notebook_02_real_data_pilot.ipynb` for a real-data run on the FRED monthly pilot dataset.